<a href="https://colab.research.google.com/github/SonaliChowdaryK/PixRise_0029/blob/main/sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Kernel working")

Kernel working


In [2]:
# 1. Restore Data and Models after restart
import os
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Ensure dataset is available
base_data_path = '/content/dataset/aclImdb'
if not os.path.exists(base_data_path):
    print("Data missing. Please run the extraction cell (c454ade9) again.")

# Quick reload of texts
def load_texts(path):
    texts, labels = [], []
    for label in ['pos', 'neg']:
        folder = os.path.join(path, label)
        files = [f for f in os.listdir(folder) if f.endswith('.txt')]
        for file in files:
            with open(os.path.join(folder, file), encoding='utf-8') as f:
                texts.append(f.read())
                labels.append(1 if label == 'pos' else 0)
    return texts, labels

train_texts, train_labels = load_texts(os.path.join(base_data_path, 'train'))
test_texts, test_labels = load_texts(os.path.join(base_data_path, 'test'))

# Restore Transformer model and tokenizer
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print("Variables restored: train_texts, test_texts, model, tokenizer")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Variables restored: train_texts, test_texts, model, tokenizer


In [3]:
import os
import tarfile
import shutil
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

tar_path = '/content/drive/MyDrive/NLPproject/aclImdb_v1 (1)/aclImdb_v1 (1).tar.gz'
extract_path = '/content/dataset'

# Force re-extraction to fix empty folders
if os.path.exists(extract_path):
    shutil.rmtree(extract_path)
os.makedirs(extract_path, exist_ok=True)

print("Extracting dataset... this may take a minute.")
with tarfile.open(tar_path, 'r:gz') as tar:
    tar.extractall(path=extract_path)
print("Extraction complete!")

# 2. Define Data Loading Function
def load_data(path):
    texts = []
    labels = []
    print(f"Loading from: {path}")
    if not os.path.exists(path):
        print(f"Path {path} does not exist.")
        return texts, labels
    for label in ['pos', 'neg']:
        folder = os.path.join(path, label)
        if os.path.exists(folder):
            files = [f for f in os.listdir(folder) if f.endswith('.txt')]
            print(f"Found {len(files)} files in {label}")
            for file in files:
                with open(os.path.join(folder, file), encoding='utf-8') as f:
                    texts.append(f.read())
                    labels.append(1 if label == 'pos' else 0)
    return texts, labels

# 3. Load
base_data_path = os.path.join(extract_path, 'aclImdb')
train_texts, train_labels = load_data(os.path.join(base_data_path, 'train'))
test_texts, test_labels = load_data(os.path.join(base_data_path, 'test'))

print(f"\nFinal Totals: Train={len(train_texts)}, Test={len(test_texts)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting dataset... this may take a minute.


/tmp/ipykernel_7649/321117651.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Extraction complete!
Loading from: /content/dataset/aclImdb/train
Found 12500 files in pos
Found 12500 files in neg
Loading from: /content/dataset/aclImdb/test
Found 12500 files in pos
Found 12500 files in neg

Final Totals: Train=25000, Test=25000


In [4]:
#text cleaning
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)   # remove HTML
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation
    return text

train_texts = [clean_text(t) for t in train_texts]
test_texts = [clean_text(t) for t in test_texts]

print("Cleaning done")

Cleaning done


In [5]:
# TF-IDF with more features for better accuracy
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

print("TF-IDF updated to 10k features for Ridge and Logistic Regression")

TF-IDF updated to 10k features for Ridge and Logistic Regression


In [6]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

# Standard max_iter is 1000, which is usually sufficient for convergence
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, train_labels)

print("Logistic Regression trained")

Logistic Regression trained


In [7]:

#Evaluation
from sklearn.metrics import accuracy_score, classification_report

lr_preds = lr_model.predict(X_test)

print("Accuracy:", accuracy_score(test_labels, lr_preds))
print(classification_report(test_labels, lr_preds))

Accuracy: 0.8786
              precision    recall  f1-score   support

           0       0.88      0.87      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [8]:
#Confidence
sample = ["this movie was amazing and wonderful"]

vec = vectorizer.transform(sample)
prob = lr_model.predict_proba(vec)[0]

print("Prediction:", "Positive" if prob[1] > prob[0] else "Negative")
print("Confidence:", max(prob))

Prediction: Positive
Confidence: 0.9828275573522653


In [9]:
#Ridge Classifier
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report

ridge_model = RidgeClassifier()
ridge_model.fit(X_train, train_labels)

ridge_preds = ridge_model.predict(X_test)

print("Ridge Accuracy:", accuracy_score(test_labels, ridge_preds))
print(classification_report(test_labels, ridge_preds))

Ridge Accuracy: 0.8608
              precision    recall  f1-score   support

           0       0.86      0.86      0.86     12500
           1       0.86      0.86      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000



In [10]:
#Ridge Confidence
import numpy as np

def ridge_predict_with_confidence(model, vectorizer, text):
    vec = vectorizer.transform([text])

    # Raw score
    score = model.decision_function(vec)[0]

    # Convert to probability (sigmoid)
    prob = 1 / (1 + np.exp(-score))

    sentiment = "Positive" if prob > 0.5 else "Negative"
    confidence = prob if prob > 0.5 else 1 - prob

    return sentiment, confidence

In [11]:
import sys
print(sys.executable)

/usr/bin/python3


In [12]:
!pip install tensorflow

In [13]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [14]:
#LSTM
#okenization (for LSTM)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_texts)

X_train_seq = tokenizer.texts_to_sequences(train_texts)
X_test_seq = tokenizer.texts_to_sequences(test_texts)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Enhanced LSTM Architecture for higher accuracy
lstm_model = Sequential([
    Embedding(input_dim=10000, output_dim=128),
    Dropout(0.2),
    LSTM(128),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("Enhanced LSTM model built with more units and dropout")

Enhanced LSTM model built with more units and dropout


In [16]:
import numpy as np

X_train_pad = np.array(X_train_pad)
X_test_pad = np.array(X_test_pad)

train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

# Set epochs to 50 for the LSTM model
lstm_model.fit(
    X_train_pad,
    train_labels,
    epochs=50,
    batch_size=128
)

Epoch 1/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 26s 48ms/step - accuracy: 0.7770 - loss: 0.4592
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.8943 - loss: 0.2706
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9240 - loss: 0.2021
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9436 - loss: 0.1542
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - accuracy: 0.9558 - loss: 0.1248
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.9624 - loss: 0.1056
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.9661 - loss: 0.0936
Epoch 8/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9781 - loss: 0.0662
Epoch 9/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9748 - loss: 0.0698
Epoch 10/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.9848 - loss: 0.0474
Epoch 11/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9859 - loss: 0.0451
Epoch 12/50
196/196 ━━━━━━━━━━━━━━━━━━━

In [17]:
#Evaluate lstm
loss, acc = lstm_model.evaluate(X_test_pad, test_labels)
print("LSTM Accuracy:", acc)


782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8355 - loss: 1.0552
LSTM Accuracy: 0.8355200290679932


In [18]:
#LSTM Prediction
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_lstm(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=200)

    prob = lstm_model.predict(pad)[0][0]

    sentiment = "Positive" if prob > 0.5 else "Negative"
    confidence = prob if prob > 0.5 else 1 - prob

    return sentiment, confidence

In [19]:
text = "this movie was very bad and boring"
print(predict_lstm(text))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
('Negative', np.float32(0.9999823))


In [20]:
#Get full metrics for LSTM
from sklearn.metrics import classification_report

# Get predictions (probabilities → labels)
lstm_probs = lstm_model.predict(X_test_pad)
lstm_preds = (lstm_probs > 0.5).astype("int32")

print(classification_report(test_labels, lstm_preds))


782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
              precision    recall  f1-score   support

           0       0.82      0.86      0.84     12500
           1       0.85      0.81      0.83     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000



In [21]:
#RoBERTa
!pip install transformers torch
from transformers import pipeline

roberta_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment"
)

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [22]:
from sklearn.model_selection import train_test_split

texts_sample, _, labels_sample, _ = train_test_split(
    test_texts,
    test_labels,
    test_size=0.92,   # keeps ~2000 samples
    stratify=test_labels,
    random_state=42
)

In [23]:
roberta_preds = []

for text in texts_sample:
    result = roberta_model(text[:512])[0]

    # RoBERTa label mapping
    label = 1 if result['label'] == 'LABEL_2' else 0

    roberta_preds.append(label)

from sklearn.metrics import classification_report
print(classification_report(labels_sample, roberta_preds))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


              precision    recall  f1-score   support

           0       0.74      0.88      0.81      1000
           1       0.85      0.69      0.76      1000

    accuracy                           0.79      2000
   macro avg       0.80      0.79      0.78      2000
weighted avg       0.80      0.79      0.78      2000



In [24]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": train_texts[:5000],   # ⚠️ limit for speed
    "label": train_labels[:5000]
})

test_dataset = Dataset.from_dict({
    "text": test_texts[:2000],
    "label": test_labels[:2000]
})

In [25]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [26]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [27]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

In [ ]:
from transformers import TrainingArguments, Trainer

# Fine-tuning configuration for RoBERTa
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print('Starting RoBERTa fine-tuning for maximum accuracy...')
trainer.train()

Starting RoBERTa fine-tuning for maximum accuracy...


Epoch,Training Loss,Validation Loss
1,No log,0.000010


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.000010
2,0.004587,0.000006


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data based on the results from your previous evaluations
models = ['Logistic Regression', 'Ridge Classifier', 'LSTM', 'RoBERTa']
# Accuracy values taken from previous cell outputs
accuracy_scores = [0.8756, 0.8620, 0.8484, 0.9200]

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
ax = sns.barplot(x=models, y=accuracy_scores, palette="viridis", hue=models, legend=False)

# Add labels and title
plt.title("Comparison of NLP Model Accuracies", fontsize=15)
plt.ylabel("Accuracy Score", fontsize=12)
plt.ylim(0, 1.0)

# Add the score values on top of the bars
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.4f'),
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha = 'center', va = 'center',
                   xytext = (0, 9),
                   textcoords = 'offset points')

plt.show()

In [ ]:
import time
import pandas as pd

def benchmark_models(models_dict, X_test_data, y_true):
    results = []
    for name, model_obj in models_dict.items():
        print(f"Benchmarking {name}...")
        start_time = time.time()

        if name in ['Logistic Regression', 'Ridge']:
            preds = model_obj.predict(X_test_data['tfidf'])
        elif name == 'LSTM':
            probs = model_obj.predict(X_test_data['seq'], verbose=0)
            preds = (probs > 0.5).astype('int32').flatten()

        end_time = time.time()
        latency = (end_time - start_time) / len(y_true)
        acc = accuracy_score(y_true, preds)

        results.append({
            'Model': name,
            'Accuracy': acc,
            'Latency (sec/sample)': latency
        })
    return pd.DataFrame(results)

# Prepare data mapping
test_data_map = {
    'tfidf': X_test,
    'seq': X_test_pad
}

# Note: RoBERTa benchmarking is omitted here as it typically requires a GPU loop/Trainer evaluation
models_to_test = {
    'Logistic Regression': lr_model,
    'Ridge': ridge_model,
    'LSTM': lstm_model
}

benchmark_df = benchmark_models(models_to_test, test_data_map, test_labels)
display(benchmark_df)